In [1]:
conda install pandas tqdm pydantic python-dotenv jupyter -y

Retrieving notices: ...working... done
Solving environment: done


==> WARNING: A newer version of conda exists. <==
  current version: 23.9.0
  latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=26.1.1



## Package Plan ##

  environment location: /Users/IvyGuo/anaconda3

  added / updated specs:
    - jupyter
    - pandas
    - pydantic
    - python-dotenv
    - tqdm


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    jupyter-1.1.1              |  py311hca03da5_0           9 KB
    python-dotenv-1.2.1        |  py311hca03da5_0          51 KB
    tqdm-4.67.3                |  py311h7eb115d_1         159 KB
    ------------------------------------------------------------
                                           Total:         219 KB

The foll

In [3]:
pip install --upgrade openai

  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/c9/30/844dc675ee6902579b8eef01ed23917cc9319a1c9c0c14ec6e39340c96d0/openai-2.24.0-py3-none-any.whl.metadata
  Obtaining dependency information for jiter<1,>=0.10.0 from https://files.pythonhosted.org/packages/50/f6/566364c777d2ab450b92100bea11333c64c38d32caf8dc378b48e5b20c46/jiter-0.13.0-cp311-cp311-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.6.1
    Uninstalling jiter-0.6.1:
      Successfully uninstalled jiter-0.6.1
  Attempting uninstall: openai
    Found existing installation: openai 1.52.1
    Uninstalling openai-1.52.1:
      Successfully uninstalled openai-1.52.1
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import time
from typing import List, Literal, Optional

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field

from openai import OpenAI

DATA_PATH = "/Users/IvyGuo/Documents/GitHub/AIL-sensory-fair/Sensory v1  - sensoryfair_v1.csv" 
OUT_JSONL = "llm_tagged_data_v1.jsonl"
OUT_CSV = "llm_tagged_data_v1.csv"

MODEL = "gpt-4o-mini" 
TEMPERATURE = 0

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set in your environment."

client = OpenAI()
print("Setup OK. Data path:", DATA_PATH)

Setup OK. Data path: /Users/IvyGuo/Documents/GitHub/AIL-sensory-fair/Sensory v1  - sensoryfair_v1.csv


In [4]:
df = pd.read_csv(DATA_PATH)
print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(3)

Rows: 52
Columns: ['product_id', 'url_hash', 'source', 'scraped_at', 'product_name', 'product_url', 'short_description', 'full_description', 'category', 'text_for_labeling', 'seamless', 'fabric_softness', 'wool_present', 'texture_level', 'pressure_level', 'adaptive_or_sensory_claim', 'seamless_evidence', 'fabric_softness_evidence', 'wool_evidence', 'texture_evidence', 'pressure_evidence', 'adaptive_claim_evidence', 'labeler', 'label_status', 'review_notes']


,product_id,url_hash,source,scraped_at,product_name,product_url,short_description,full_description,category,text_for_labeling,...,adaptive_or_sensory_claim,seamless_evidence,fabric_softness_evidence,wool_evidence,texture_evidence,pressure_evidence,adaptive_claim_evidence,labeler,label_status,review_notes
0,1,a5debd253afe,sensoryfair.com,NaN,Abby & Noah Sensory Compression Vest for Women...,https://sensoryfair.com/product/abby-noah-sens...,Therapist-Designed Compression Hug: Our “weara...,Therapist-Designed Compression Hug: Our “weara...,NaN,product_name: Abby & Noah Sensory Compression ...,...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unlabeled,NaN
1,2,eae004a58e62,sensoryfair.com,NaN,Snugabye Adaptive Back Zip Convert-A-Foot Slee...,https://sensoryfair.com/product/adaptive-back-...,NaN,This collection was designed with your special...,NaN,product_name: Snugabye Adaptive Back Zip Conve...,...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unlabeled,NaN
2,3,e2520a1845e7,sensoryfair.com,NaN,"CBObaby Adaptive Bodysuit, Short Sleeve, Front...",https://sensoryfair.com/product/adaptive-bodys...,NaN,Discover comfort and functionality with this i...,NaN,"product_name: CBObaby Adaptive Bodysuit, Short...",...,Unknown,NaN,NaN,NaN,NaN,NaN,NaN,NaN,unlabeled,NaN


In [6]:
# -----------------------------
# 1) Define the structured output schema (Pydantic)
# -----------------------------

Tri = Literal["yes", "no", "unknown"]
Quad = Literal["low", "medium", "high", "unknown"]
Overall = Literal["yes", "no", "unclear"]

class LabeledFeature(BaseModel):
    value: Literal["yes","no","unknown","low","medium","high"] = Field(..., description="Feature label.")
    evidence: List[str] = Field(default_factory=list, description="Verbatim quotes supporting the label (required if not unknown).")

class CoreFeatures(BaseModel):
    seamless: LabeledFeature
    fabric_softness: LabeledFeature
    wool_present: LabeledFeature
    texture_level: LabeledFeature
    pressure_level: LabeledFeature
    adaptive_or_sensory_claim: LabeledFeature

class ExtendedFeatures(BaseModel):
    closure_type: Optional[dict] = None
    potential_irritants: Optional[dict] = None
    sensory_support_features: Optional[dict] = None

class OverallLabel(BaseModel):
    value: Overall
    reason: str

class LLMTagRecord(BaseModel):
    product_id: str
    core_features: CoreFeatures
    # keep extended optional for now; you can enable later
    overall_sensory_friendly: OverallLabel
    sensory_domains_supported: List[Literal["tactile","auditory","visual","olfactory"]] = Field(default_factory=list)
    sensory_domains_risk: List[Literal["tactile","auditory","visual","olfactory"]] = Field(default_factory=list)
    confidence: float = Field(..., ge=0.0, le=1.0)
    review_notes: str = ""

# Helper: map each feature to its allowed label set.
# We'll enforce allowed values in the prompt, and validate with Pydantic.


In [8]:
# -----------------------------
# 2) Prompt template (rules-first, evidence-required)
# -----------------------------

SYSTEM_INSTRUCTIONS = """You are a strict sensory-friendly product labeling assistant.
Your job is to extract sensory-relevant features from product descriptions.

CRITICAL RULES:
1) NO INFERENCE: Only label based on explicit text. Do not use outside knowledge.
2) EVIDENCE REQUIRED: Every non-unknown feature label MUST include at least one verbatim quote from the description.
3) CONTRADICTIONS: If equally strong conflicting signals exist, label as unknown and record both phrases in review_notes.
4) PRECEDENCE: technical terms > strong modifiers > fit/style descriptors > generic comfort words.
5) DO NOT GUESS: If missing, use unknown.
6) Output MUST match the provided schema exactly (valid JSON)."""

LABEL_GUIDE = """SENSORY-FRIENDLY CONTEXT (REFERENCE):
- Sensory reactivity differences can occur across multiple sensory domains (e.g., touch, vision, sound).
- Individuals may be hyperreactive (avoid) or seek sensory input; product text must explicitly mention claims/features.
TACTILE SUPPORT EXAMPLES: seamless/no seams, flat seams/flatlock, tagless, ultra/super/buttery soft, brushed interior, plush, stretchy fabrics, relaxed/loose fit.
TACTILE RISK EXAMPLES: wool/merino/wool blend, stiff/structured/firm/heavy canvas, ribbed/waffle/cable knit/embossed/raised pattern (texture), strong/firm/graduated compression.
AUDITORY SUPPORT EXAMPLES: noise reducing, quiet, sound dampening, calming sounds.
VISUAL SUPPORT EXAMPLES: dim light, neutral colors, avoids flicker.
OLFACTORY SUPPORT EXAMPLES: unscented, mild scent; strong scents can be overwhelming.
IMPORTANT: Only label when the description explicitly supports it."""

# Decision-tree hints (from your v1 rules) — included to improve consistency:
DECISION_HINTS = """FEATURE DECISION HINTS:
SEAMLESS:
- yes if: seamless, no seams, no side seams, flatlock seams, flat seams, bonded seams, bonded construction (re seam removal)
- no if: reinforced seams, double stitched seams, exposed seams
FABRIC_SOFTNESS:
- high if: ultra soft, super soft, buttery soft, extra soft, brushed interior, plush
- medium if: soft, comfortable, smooth fabric
- low if: stiff, structured fabric, firm material, heavy canvas, durable canvas
WOOL_PRESENT:
- yes if: wool, merino wool, wool blend
- no if: full material composition lists other fibers and excludes wool (e.g., 100% cotton)
TEXTURE_LEVEL:
- high if: ribbed, waffle knit, cable knit, textured, raised pattern, embossed, embroidered texture
- medium if: knit (visible knit), patterned, heathered
- low if: smooth, flat surface, no texture
PRESSURE_LEVEL:
- high if: firm compression, strong compression, compression therapy, graduated compression
- medium if: compression, supportive, snug fit, body-hugging, slim fit, form-fitting
- low if: loose fit, relaxed fit, flowy, oversized
ADAPTIVE_OR_SENSORY_CLAIM:
- yes if: sensory friendly, autism friendly, adaptive clothing, special needs clothing, designed for sensory comfort
- no if: explicitly not adaptive/sensory-focused"""

def build_user_prompt(product: dict) -> str:
    name = product.get("product_name","") or ""
    short_desc = product.get("short_description","") or ""
    full_desc = product.get("full_description","") or ""
    url = product.get("product_url","") or ""
    price = str(product.get("price","") or "")

    return f"""{LABEL_GUIDE}

{DECISION_HINTS}

PRODUCT:
name: {name}
url: {url}
price: {price}

short_description:
{short_desc}

full_description:
{full_desc}

Return JSON that matches the schema exactly.
"""


In [10]:
# -----------------------------
# 3) Data helpers
# -----------------------------

def get_text_fields(row: pd.Series) -> dict:
    # Adapt these keys if your CSV differs.
    return {
        "product_name": row.get("product_name", ""),
        "short_description": row.get("short_description", ""),
        "full_description": row.get("full_description", ""),
        "product_url": row.get("product_url", ""),
        "price": row.get("price", ""),
    }

def coerce_product_id(row: pd.Series, idx: int) -> str:
    # Prefer a stable ID if you have one; fallback to index.
    for k in ["product_id", "id", "sku", "handle"]:
        if k in row and pd.notna(row[k]):
            return str(row[k])
    return str(idx)


In [12]:
# -----------------------------
# 4) OpenAI call (Structured Outputs via Pydantic parse)
# -----------------------------
# Uses Responses.parse(...) pattern from Structured Outputs docs citeturn1view1

def label_one(product: dict, product_id: str, *, max_retries: int = 3, sleep_s: float = 1.0) -> LLMTagRecord:
    user_prompt = build_user_prompt(product)

    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.responses.parse(
                model=MODEL,
                instructions=SYSTEM_INSTRUCTIONS,
                input=[
                    {"role": "user", "content": [{"type": "input_text", "text": user_prompt}]}
                ],
                text_format=LLMTagRecord,
                temperature=TEMPERATURE,
            )
            rec: LLMTagRecord = resp.output_parsed
            # ensure product_id matches our local ID
            rec.product_id = product_id
            return rec
        except Exception as e:
            last_err = e
            time.sleep(sleep_s * attempt)

    raise RuntimeError(f"Failed after {max_retries} attempts: {last_err}")


In [14]:
# -----------------------------
# 5) Batch labeling + save outputs
# -----------------------------

# Tip: start small for a pilot run
N_LIMIT = None  # e.g., 50 for testing; None = all rows

rows = df if N_LIMIT is None else df.head(N_LIMIT)

jsonl_lines = []
records_for_df = []

for idx, row in tqdm(list(rows.iterrows()), total=len(rows)):
    prod = get_text_fields(row)
    pid = coerce_product_id(row, idx)

    try:
        labeled = label_one(prod, pid)
        as_dict = labeled.model_dump()
        jsonl_lines.append(json.dumps(as_dict, ensure_ascii=False))
        records_for_df.append({
            "product_id": pid,
            "llm_tags_json": json.dumps(as_dict, ensure_ascii=False),
            "overall_sensory_friendly": as_dict["overall_sensory_friendly"]["value"],
            "confidence": as_dict["confidence"],
        })
    except Exception as e:
        # Save failures so you can re-run them later
        err_obj = {
            "product_id": pid,
            "error": str(e),
            "product_name": prod.get("product_name",""),
            "product_url": prod.get("product_url",""),
        }
        jsonl_lines.append(json.dumps(err_obj, ensure_ascii=False))
        records_for_df.append({
            "product_id": pid,
            "llm_tags_json": "",
            "overall_sensory_friendly": "",
            "confidence": None,
            "error": str(e)
        })

# Write files
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    f.write("\n".join(jsonl_lines))

out_df = pd.DataFrame(records_for_df)
out_df.to_csv(OUT_CSV, index=False)

print("Wrote:", OUT_JSONL, "and", OUT_CSV)
out_df.head(10)

  0%|          | 0/52 [00:00<?, ?it/s]

Wrote: llm_tagged_data_v1.jsonl and llm_tagged_data_v1.csv


,product_id,llm_tags_json,overall_sensory_friendly,confidence,error
0,1,,,None,Failed after 3 attempts: Error code: 429 - {'e...
1,2,,,None,Failed after 3 attempts: Error code: 429 - {'e...
2,3,,,None,Failed after 3 attempts: Error code: 429 - {'e...
3,4,,,None,Failed after 3 attempts: Error code: 429 - {'e...
4,5,,,None,Failed after 3 attempts: Error code: 429 - {'e...
5,6,,,None,Failed after 3 attempts: Error code: 429 - {'e...
6,7,,,None,Failed after 3 attempts: Error code: 429 - {'e...
7,8,,,None,Failed after 3 attempts: Error code: 429 - {'e...
8,9,,,None,Failed after 3 attempts: Error code: 429 - {'e...
9,10,,,None,Failed after 3 attempts: Error code: 429 - {'e...
